In [2]:
from src.server.models.dms_model import DmsModel
from src.db import DbConnection
from src.runtime_constants import rr
import math


conn = DbConnection(rr.ENV_PROD, True)

guid = "c51ef766-516d-45df-a01a-9fba927fcb23"

df = DmsModel.read_df_from_external_data(conn, guid=guid)
#conn.close()
df

,Title,Publisher,Bundle Id GP,Bundle Id Ios,Published GP,Published Ios,Countries GP,Rating GP,Countries Ios,Rating Ios,...,Users12M GP WW,Users12M GP US,Users1M GP WW,Users1M GP US,Users12M Ios WW,Users12M Ios US,Users1M Ios WW,Users1M Ios US,D28 retention GP,D28 retention Ios
0,劲舞团 - 以团之名！舞团大赛一触即发,网易移动游戏,None,com.netease.jwt,None,2016-09-13,None,NaN,"[CN, HK, SG, TW, US]",4.649408,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,仙命決——俠侶雙修，道友PVP對決,Entrepreneur Game,com.immortals.tw,com.immortals.ios.tw,2019-08-16,2019-08-21,"[AE, AM, AR, AT, AU, AZ, BA, BD, BE, BG, BH, B...",4.562818,"[AE, AR, AT, AU, BE, BR, CA, CH, CL, CN, CZ, D...",4.719011,...,3006017.0,373279.0,191530.0,415743.0,2437851.0,1118437.0,57242.0,78121.0,0.140072,0.127267
2,Dungeon & Heroes: 3D RPG,DroidHen,com.droidhen.dungeon,com.droidhen.dungeon,2018-02-02,2018-04-21,"[AE, AM, AR, AT, AU, AZ, BA, BD, BE, BG, BH, B...",4.203316,"[AE, AR, AT, AU, BE, BR, CA, CH, CL, CN, CZ, D...",4.301443,...,160180.0,22085.0,107258.0,84916.0,16492.0,NaN,46838.0,33553.0,NaN,NaN
3,Fight For America: Country War,HOMA GAMES,None,com.Hitica.FightForAmerica,None,2023-01-26,None,NaN,"[AE, AT, AU, BE, BR, CA, CH, CL, CZ, DE, DK, E...",4.490494,...,NaN,NaN,NaN,NaN,4177070.0,4998376.0,5999.0,882036.0,NaN,0.058975
4,My Horse,"MunkyFun, Inc.",None,com.naturalmotion.h0rs3,None,2011-09-15,None,NaN,"[AE, AR, AT, AU, BE, BR, CA, CH, CL, CN, CZ, D...",4.567333,...,NaN,NaN,NaN,NaN,1908740.0,1048184.0,131221.0,131221.0,NaN,0.179283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9443,Cross Stitch Masters,TAPCLAP,com.tapclap.crossstitch,None,2019-12-18,None,"[AE, AM, AR, AT, AU, AZ, BA, BE, BG, BH, BO, B...",4.134021,None,NaN,...,3631591.0,413836.0,114708.0,2590.0,NaN,NaN,NaN,NaN,NaN,NaN
9444,Real Guitar - Music Band Game,Gismart,com.gismart.guitar,None,2013-05-17,None,"[AE, AM, AR, AT, AU, AZ, BA, BD, BE, BG, BH, B...",3.587494,None,NaN,...,15277292.0,453138.0,1272638.0,81487.0,NaN,NaN,NaN,NaN,NaN,NaN
9445,Word Universe,FIOGONIA LIMITED,com.fbig.words,None,2022-11-25,None,"[AE, AM, AR, AU, AZ, BA, BD, BH, BR, CA, CN, C...",4.601919,None,NaN,...,1430992.0,553870.0,144.0,145.0,NaN,NaN,NaN,NaN,0.104072,NaN
9446,Bricks Ball Puzzle,PoseidonGames,com.iposedon.lite.bbricksball,None,2021-05-14,None,"[AE, AM, AR, AZ, BA, BD, BG, BH, BO, BR, BY, C...",3.847658,None,NaN,...,22444081.0,57588.0,3841.0,12321.0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
conn.query("""
DROP TABLE IF EXISTS scraped_data.ext_data
""")
conn.query("""
CREATE TABLE scraped_data.ext_data (
app_id_int mediumint default 0 not null,
app_id_in_store tinytext not null,
downloads_gp_us bigint default -1 not null,
downloads_gp_ww bigint default -1 not null,
downloads_12m_gp_us bigint default -1 not null,
downloads_12m_gp_ww bigint default -1 not null,
d28_ret_gp decimal(6, 2) default -1 not null,
revenues_gp_us bigint default -1 not null,
revenues_gp_ww bigint default -1 not null,
revenues_12m_gp_us int default -1 not null,
revenues_12m_gp_ww int default -1 not null
)
""")
conn.commit()
app_ids = []
bulk_data = []
for index, row in df.iterrows():
    app_id_in_store = row["Bundle Id GP"]
    if app_id_in_store is None:
        continue
    if app_id_in_store in app_ids:
        continue
    app_ids.append(app_id_in_store)
    def float_safe(val: str) -> float:
        res = -1
        try:
            res = float(val)
            if math.isnan(res):
                res = -1
        except:
            pass
        return res
    def int_safe(val: str) -> int:
        res = -1
        try:
            res = int(val)
        except:
            pass
        return res
    down_gp_us = int_safe(row["Downloads GP US"])
    down_gp_ww = int_safe(row["Downloads GP WW"])
    down_12m_gp_us = int_safe(row["Downloads12M GP US"])
    down_12m_gp_ww = int_safe(row["Downloads12M GP WW"])
    d28_ret_gp = round(100 * float_safe(row["D28 retention GP"]), 2)
    revenues_12m_gp_us = int_safe(row["Revenues12M GP US"])
    revenues_12m_gp_ww = int_safe(row["Revenues12M GP WW"])
    revenues_gp_us = int_safe(row["Revenues GP US"])
    revenues_gp_ww = int_safe(row["Revenues GP WW"])

    bulk_data.append((app_id_in_store, down_gp_us, down_gp_ww, down_12m_gp_us, down_12m_gp_ww, d28_ret_gp, revenues_12m_gp_us, revenues_12m_gp_ww, revenues_gp_us, revenues_gp_ww))

conn.bulk("""
INSERT INTO scraped_data.ext_data (app_id_int, app_id_in_store, downloads_gp_us, downloads_gp_ww, downloads_12m_gp_us, downloads_12m_gp_ww, d28_ret_gp, revenues_12m_gp_us, revenues_12m_gp_ww, revenues_gp_us, revenues_gp_ww) VALUES (0, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", bulk_data)

conn.query("""
UPDATE scraped_data.ext_data e
INNER JOIN app.map_app_id m ON m.app_id = lower(e.app_id_in_store)
SET e.app_id_int = m.id
""")

conn.query("""
delete from scraped_data.ext_data where app_id_int = 0;
""")

conn.query("""
alter table scraped_data.ext_data
    add constraint ext_data_pk
        primary key (app_id_int);
""")

conn.commit()

In [4]:
conn.close()